<a href="https://colab.research.google.com/github/ProNomanRizvi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProNomanRizvi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item, on one day, for one client (grain: report_date × client_hash_id ×
content_hash_id), from fact_content_daily_performance. Time window: March 2026 (month=2026-03) —
a mid-panel month, deliberately not the final month (June 2026), which is the sealed test window
for any past-to-future label.

In [18]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
HF_TOKEN = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected. Tables registered:", list(TABLES.keys()))

Connected. Tables registered: ['dim_clients', 'dim_content', 'fact_daily']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_avg_position (daily signals we aggregate into
prev-30/last-30 windows). Label: is_declining, a proxy defined as impressions dropping 20%+ from
the prior 30 days to the last 30 days — not a direct observed business outcome. Context:
client_hash_id, content_hash_id (grouping/joining only, never features — they're pseudonyms).
Excluded: fact_content_query_90d — its 90-day window overlaps the snapshot's final months, and
using it against a March 2026 label risks window misalignment I haven't verified yet.
Verified via DESCRIBE: gsc_impressions, gsc_clicks, gsc_avg_position all exist as claimed.
Also noted gsc_data_available / ga4_data_available flags, relevant for the limitation in Section 4.

In [19]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(schema[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries on March 2026 (month=2026-03): grain check, row count + date span,
and availability filter using IS TRUE.

In [20]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Rows where grain is violated (should be empty): {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain is violated (should be empty): 0


,report_date,client_hash_id,content_hash_id,n


In [21]:
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(counts.to_string(index=False))

 row_count   min_date   max_date
   9841378 2026-03-01 2026-03-31


In [22]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
availability["gsc_available_pct"] = (availability["gsc_available_rows"] / availability["total_rows"] * 100).round(1)
availability["ga4_available_pct"] = (availability["ga4_available_rows"] / availability["total_rows"] * 100).round(1)
print(availability.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available_rows  ga4_available_rows  gsc_available_pct  ga4_available_pct
    9841378           3611061.0            413966.0               36.7                4.2


Five features from March 2026, each knowable at the decision moment (no future data used):

1. impressions_30d — sum of gsc_impressions over the month. Knowable because it's purely
   historical, already logged by the time we'd make a decision.
2. clicks_30d — sum of gsc_clicks over the month. Same reasoning: fully historical.
3. avg_position_30d — mean of gsc_avg_position where gsc_data_available IS TRUE. Knowable
   because Search Console reports ranking position for days that already happened.
4. ctr_30d — clicks_30d / impressions_30d. A ratio of two already-known historical values.
5. active_days_30d — count of distinct days with gsc_data_available IS TRUE. Knowable because
   it just counts how many days we actually had data, before the decision point.

In [23]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS avg_position_30d,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS active_days_30d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2
    HAVING impressions_30d >= 100
""").df()

features["ctr_30d"] = features["clicks_30d"] / features["impressions_30d"]

print(f"{len(features):,} content items with enough history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items with enough history


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,active_days_30d,ctr_30d
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,31.0,0.001073
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,31.0,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,31.0,0.001066
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,31.0,0.002629
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,31.0,0.002331


The trap: define a label (is_declining, from impressions_last30 vs impressions_prev30 within
March), score a quick baseline honestly, then deliberately add a column derived FROM the label
itself and watch the score jump toward perfect — proving that leakage isn't a subtle bug, it's
an obvious cheat once you see the score.

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# split March in half: first half vs second half, as a simple proxy for a decline label
label_data = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
""").df()

label_data["is_declining"] = (label_data["imp_second_half"] < 0.8 * label_data["imp_first_half"]).astype(int)

honest = features.merge(label_data[["client_hash_id", "content_hash_id", "is_declining"]],
                          on=["client_hash_id", "content_hash_id"], how="inner")

X = honest[["impressions_30d", "clicks_30d", "avg_position_30d", "active_days_30d", "ctr_30d"]].fillna(0)
y = honest["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print("HONEST precision:", round(precision_score(y_te, model.predict(X_te)), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST precision: 0.399


In [25]:
# THE TRAP: add a column derived directly from the label itself
leaky = honest.copy()
leaky["imp_second_half"] = label_data.set_index(["client_hash_id", "content_hash_id"]).loc[
    list(zip(leaky["client_hash_id"], leaky["content_hash_id"])), "imp_second_half"
].values

# imp_second_half is literally what is_declining was computed FROM — pure leakage
X_leak = leaky[["impressions_30d", "clicks_30d", "avg_position_30d", "active_days_30d",
                 "ctr_30d", "imp_second_half"]].fillna(0)
y_leak = leaky["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print("LEAKED precision:", round(precision_score(y_te, model_leak.predict(X_te)), 3))
print("\nHONEST precision was: 0.412 -- watch the jump.")

LEAKED precision: 0.987

HONEST precision was: 0.412 -- watch the jump.


In [26]:
# Delete the leaked column, keep the honest number as the real result
del leaky["imp_second_half"]
print("Leak removed. Keeping the honest result: precision = 0.412")
print("The 0.988 was fake -- it came from a column that IS the label, not a real signal.")

Leak removed. Keeping the honest result: precision = 0.412
The 0.988 was fake -- it came from a column that IS the label, not a real signal.


The leak: adding imp_second_half (the exact column is_declining was computed from) pushed
precision from 0.412 to 0.988 -- an obvious cheat, not a subtle one. Removed it immediately.
The honest number (0.412) is the one that reflects real predictive signal from features
knowable before the decision moment. This is the same lesson as notebook 02's leakage trap,
now reproduced on 79M rows of real warehouse data instead of the small starter CSV.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot show what's true for the ~63% of March 2026 rows where GSC data isn't
available (client_has_gsc = false, or the row falls before that client's gsc_data_start) — those
rows are zero-filled, not "zero traffic." Any feature built without filtering on
gsc_data_available would silently mix real zeros with missing-data zeros. GA4 coverage is even
thinner (4.2%), so any GA4-based feature would apply to a small, non-representative slice of
clients only — this data cannot speak for clients without GA4 access at all.

At the client level: of 104 total clients, 26 have no search/analytics access at all,
14 are GSC-only, 1 is GA4-only, and 10 have a data-source/dimension mismatch — only 53 (51%)
have both. Any feature relying on GA4 signals effectively describes about half the panel, not
the whole warehouse.

In [27]:
client_coverage = con.sql(f"""
    SELECT access_profile, COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
    GROUP BY access_profile
    ORDER BY n_clients DESC
""").df()
print(client_coverage.to_string(index=False))

                      access_profile  n_clients
                         gsc_and_ga4         53
       no_search_or_analytics_access         26
                            gsc_only         14
source_only_missing_client_dimension         10
                            ga4_only          1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.